# 🧠 MNIST Digit Classification
## Fully Connected Neural Network vs Convolutional Neural Network

In this notebook we build **two models** to classify handwritten digits from the MNIST dataset:
1. **Dense (Fully Connected) Neural Network** — treats each pixel as an independent feature
2. **Convolutional Neural Network (CNN)** — exploits 2-D spatial structure of images

We then compare their accuracy and discuss why CNNs are better suited for image data.

---

## 📦 Step 0 — Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

print(f"TensorFlow  : {tf.__version__}")
print(f"Keras       : {keras.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

---
## 📥 Step 1 — Load the MNIST Dataset

In [ ]:
# Load MNIST — automatically downloaded & cached by Keras
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = keras.datasets.mnist.load_data()

print("─── Raw dataset shapes ───────────────────────────")
print(f"X_train : {X_train_raw.shape}   ← 60 000 images of 28×28 pixels")
print(f"y_train : {y_train_raw.shape}   ← 60 000 integer labels (0–9)")
print(f"X_test  : {X_test_raw.shape}    ← 10 000 images of 28×28 pixels")
print(f"y_test  : {y_test_raw.shape}    ← 10 000 integer labels (0–9)")
print(f"\nPixel value range : {X_train_raw.min()} – {X_train_raw.max()}")
print(f"Unique labels     : {sorted(set(y_train_raw))}")

In [ ]:
# Visualize a sample of 10 digits from the training set
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Sample MNIST digits (one per class)', fontsize=13, fontweight='bold')

for digit in range(10):
    idx = np.where(y_train_raw == digit)[0][0]   # first occurrence of each digit
    axes[0, digit].imshow(X_train_raw[idx], cmap='gray')
    axes[0, digit].set_title(f'Label: {digit}', fontsize=9)
    axes[0, digit].axis('off')

# Second row — random samples
rng = np.random.default_rng(42)
rand_idx = rng.choice(len(X_train_raw), 10, replace=False)
for col, idx in enumerate(rand_idx):
    axes[1, col].imshow(X_train_raw[idx], cmap='gray')
    axes[1, col].set_title(f'{y_train_raw[idx]}', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

---
## ⚙️ Step 2 — Preprocess Data for the Fully Connected Network

A Dense layer expects a **1-D feature vector**, so we must:
1. **Flatten** each 28×28 image → 784 values
2. **Normalise** pixels to [0, 1] by dividing by 255
3. **One-hot encode** the integer labels → 10-element vectors

In [ ]:
# ── 1. Flatten ─────────────────────────────────────────────────────────
X_train_flat = X_train_raw.reshape(X_train_raw.shape[0], -1)   # (60000, 784)
X_test_flat  = X_test_raw.reshape(X_test_raw.shape[0],  -1)    # (10000, 784)

# ── 2. Normalise ────────────────────────────────────────────────────────
X_train_flat = X_train_flat.astype('float32') / 255.0
X_test_flat  = X_test_flat.astype('float32')  / 255.0

# ── 3. One-hot encode ───────────────────────────────────────────────────
y_train_ohe = to_categorical(y_train_raw, num_classes=10)
y_test_ohe  = to_categorical(y_test_raw,  num_classes=10)

print("─── Preprocessed shapes for Dense model ──────────")
print(f"X_train_flat : {X_train_flat.shape}  (28×28 → 784)")
print(f"X_test_flat  : {X_test_flat.shape}")
print(f"y_train_ohe  : {y_train_ohe.shape}   (integer → one-hot)")
print(f"\nSample label before OHE : {y_train_raw[0]}")
print(f"Sample label after  OHE : {y_train_ohe[0]}")

---
## 🏗️ Step 3 — Build & Train the Fully Connected Neural Network

In [ ]:
tf.random.set_seed(42)

# ── Architecture ────────────────────────────────────────────────────────
model_dense = keras.Sequential([
    # Input layer (implicit) → 784 features
    layers.Dense(512, activation='relu',    input_shape=(784,), name='dense_1'),
    layers.Dense(256, activation='relu',    name='dense_2'),
    layers.Dense(128, activation='relu',    name='dense_3'),
    layers.Dense(10,  activation='softmax', name='output')   # 10 classes
], name='Fully_Connected_NN')

# ── Compile ─────────────────────────────────────────────────────────────
model_dense.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_dense.summary()

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────
history_dense = model_dense.fit(
    X_train_flat, y_train_ohe,
    epochs          = 10,
    batch_size      = 128,
    validation_split= 0.1,    # hold out 10 % of training data for validation
    verbose         = 1
)

# ── Evaluate on test set ─────────────────────────────────────────────────
dense_loss, dense_acc = model_dense.evaluate(X_test_flat, y_test_ohe, verbose=0)
print(f"\n✅ Dense model — Test Loss: {dense_loss:.4f}  |  Test Accuracy: {dense_acc*100:.2f}%")

In [ ]:
# ── Plot Dense training history ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history_dense.history['loss']) + 1)

axes[0].plot(epochs_range, history_dense.history['loss'],     label='Train Loss',  color='steelblue', lw=2)
axes[0].plot(epochs_range, history_dense.history['val_loss'], label='Val Loss',    color='tomato',    lw=2, ls='--')
axes[0].set_title('Dense Model — Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history_dense.history['accuracy'],     label='Train Acc', color='steelblue', lw=2)
axes[1].plot(epochs_range, history_dense.history['val_accuracy'], label='Val Acc',   color='tomato',    lw=2, ls='--')
axes[1].set_title('Dense Model — Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Fully Connected NN — Test Accuracy: {dense_acc*100:.2f}%',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## ⚙️ Step 4 — Preprocess Data for the CNN

A `Conv2D` layer expects input shape **(height, width, channels)**. For grayscale images that means adding a channel dimension: 28×28 → 28×28×1.

In [ ]:
# ── 1. Reshape: add channel dimension ────────────────────────────────────
X_train_cnn = X_train_raw.reshape(-1, 28, 28, 1)   # (60000, 28, 28, 1)
X_test_cnn  = X_test_raw.reshape(-1, 28, 28, 1)    # (10000, 28, 28, 1)

# ── 2. Normalise ─────────────────────────────────────────────────────────
X_train_cnn = X_train_cnn.astype('float32') / 255.0
X_test_cnn  = X_test_cnn.astype('float32')  / 255.0

# ── 3. One-hot encode (same as before) ───────────────────────────────────
# y_train_ohe and y_test_ohe are already computed in Step 2

print("─── Preprocessed shapes for CNN ──────────────────")
print(f"X_train_cnn : {X_train_cnn.shape}  (28×28×1 — height, width, channels)")
print(f"X_test_cnn  : {X_test_cnn.shape}")
print(f"y_train_ohe : {y_train_ohe.shape}")
print(f"Pixel value range after normalisation: {X_train_cnn.min():.1f} – {X_train_cnn.max():.1f}")

---
## 🏗️ Step 5 — Build & Train the Convolutional Neural Network

In [ ]:
tf.random.set_seed(42)

# ── Architecture ─────────────────────────────────────────────────────────
model_cnn = keras.Sequential([

    # ── Block 1: Convolution + Pooling ───────────────────────────────────
    layers.Conv2D(
        filters=32, kernel_size=(3, 3),
        activation='relu', padding='same',
        input_shape=(28, 28, 1), name='conv1'
    ),
    layers.MaxPool2D(pool_size=(2, 2), name='pool1'),  # 28×28 → 14×14

    # ── Block 2: Convolution + Pooling ───────────────────────────────────
    layers.Conv2D(
        filters=64, kernel_size=(3, 3),
        activation='relu', padding='same', name='conv2'
    ),
    layers.MaxPool2D(pool_size=(2, 2), name='pool2'),  # 14×14 → 7×7

    # ── Block 3: Convolution (no pooling) ────────────────────────────────
    layers.Conv2D(
        filters=64, kernel_size=(3, 3),
        activation='relu', padding='same', name='conv3'
    ),

    # ── Classifier head ──────────────────────────────────────────────────
    layers.Flatten(name='flatten'),           # 7×7×64 → 3136
    layers.Dense(128, activation='relu',    name='dense_1'),
    layers.Dense(10,  activation='softmax', name='output')   # 10 classes

], name='CNN')

# ── Compile ──────────────────────────────────────────────────────────────
model_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_cnn.summary()

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
history_cnn = model_cnn.fit(
    X_train_cnn, y_train_ohe,
    epochs          = 10,
    batch_size      = 128,
    validation_split= 0.1,
    verbose         = 1
)

# ── Evaluate on test set ──────────────────────────────────────────────────
cnn_loss, cnn_acc = model_cnn.evaluate(X_test_cnn, y_test_ohe, verbose=0)
print(f"\n✅ CNN model — Test Loss: {cnn_loss:.4f}  |  Test Accuracy: {cnn_acc*100:.2f}%")

In [ ]:
# ── Plot CNN training history ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history_cnn.history['loss']) + 1)

axes[0].plot(epochs_range, history_cnn.history['loss'],     label='Train Loss', color='#7209b7', lw=2)
axes[0].plot(epochs_range, history_cnn.history['val_loss'], label='Val Loss',   color='#f72585', lw=2, ls='--')
axes[0].set_title('CNN — Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history_cnn.history['accuracy'],     label='Train Acc', color='#7209b7', lw=2)
axes[1].plot(epochs_range, history_cnn.history['val_accuracy'], label='Val Acc',   color='#f72585', lw=2, ls='--')
axes[1].set_title('CNN — Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'CNN — Test Accuracy: {cnn_acc*100:.2f}%',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 📊 Step 6 — Compare Both Models

In [ ]:
# ── Side-by-side accuracy curves ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
epochs_range = range(1, 11)

# Left: Loss comparison
axes[0].plot(epochs_range, history_dense.history['val_loss'], label='Dense val loss',
             color='steelblue', lw=2, marker='o', ms=5)
axes[0].plot(epochs_range, history_cnn.history['val_loss'],   label='CNN val loss',
             color='#7209b7',  lw=2, marker='s', ms=5)
axes[0].set_title('Validation Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Right: Accuracy comparison
axes[1].plot(epochs_range, history_dense.history['val_accuracy'], label='Dense val accuracy',
             color='steelblue', lw=2, marker='o', ms=5)
axes[1].plot(epochs_range, history_cnn.history['val_accuracy'],   label='CNN val accuracy',
             color='#7209b7',  lw=2, marker='s', ms=5)
axes[1].set_title('Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Dense NN vs CNN — Validation Metrics', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────────
labels  = ['Dense NN', 'CNN']
accs    = [dense_acc * 100, cnn_acc * 100]
losses  = [dense_loss, cnn_loss]
colors  = ['steelblue', '#7209b7']

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# Accuracy bars
bars = axes[0].bar(labels, accs, color=colors, edgecolor='black', width=0.4)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() - 0.5,
                 f'{acc:.2f}%', ha='center', va='top',
                 color='white', fontweight='bold', fontsize=13)
axes[0].set_ylim(95, 100)
axes[0].set_ylabel('Test Accuracy (%)', fontsize=12)
axes[0].set_title('Test Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# Loss bars
bars2 = axes[1].bar(labels, losses, color=colors, edgecolor='black', width=0.4)
for bar, loss in zip(bars2, losses):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.001,
                 f'{loss:.4f}', ha='center', va='bottom',
                 fontweight='bold', fontsize=12)
axes[1].set_ylabel('Test Loss', fontsize=12)
axes[1].set_title('Test Loss Comparison', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Model Performance Summary', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────
dense_params = model_dense.count_params()
cnn_params   = model_cnn.count_params()

print("\n" + "═"*62)
print(f"{'Metric':<28} {'Dense NN':>15} {'CNN':>15}")
print("═"*62)
print(f"{'Test Accuracy':<28} {dense_acc*100:>14.2f}% {cnn_acc*100:>14.2f}%")
print(f"{'Test Loss':<28} {dense_loss:>15.4f} {cnn_loss:>15.4f}")
print(f"{'Total Parameters':<28} {dense_params:>15,} {cnn_params:>15,}")
print(f"{'Input shape':<28} {'(None, 784)':>15} {'(None,28,28,1)':>15}")
print(f"{'Architecture':<28} {'Dense only':>15} {'Conv+Pool+Dense':>15}")
print("═"*62)
print(f"\n📈 CNN accuracy gain over Dense: +{(cnn_acc - dense_acc)*100:.2f} percentage points")

In [ ]:
# ── Visualise predictions: where does each model succeed / fail? ──────────
y_pred_dense = np.argmax(model_dense.predict(X_test_flat, verbose=0), axis=1)
y_pred_cnn   = np.argmax(model_cnn.predict(X_test_cnn,   verbose=0), axis=1)
y_true       = y_test_raw

# Find cases where Dense is WRONG but CNN is RIGHT
dense_wrong_cnn_right = np.where((y_pred_dense != y_true) & (y_pred_cnn == y_true))[0]

print(f"Samples Dense gets wrong but CNN gets right: {len(dense_wrong_cnn_right)}")

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('Digits Dense NN gets WRONG — CNN gets RIGHT\n(top row = image, bottom row = Dense pred | CNN pred | True label)',
             fontsize=11, fontweight='bold')

sample = dense_wrong_cnn_right[:10]
for col, idx in enumerate(sample):
    axes[0, col].imshow(X_test_raw[idx], cmap='gray')
    axes[0, col].axis('off')
    axes[1, col].text(0.5, 0.5,
                      f'Dense:{y_pred_dense[idx]}\nCNN:{y_pred_cnn[idx]}\nTrue:{y_true[idx]}',
                      ha='center', va='center', fontsize=8.5,
                      color='tomato' if y_pred_dense[idx] != y_true[idx] else 'green')
    axes[1, col].axis('off')

plt.tight_layout()
plt.show()

---
## 📝 Analysis & Key Takeaways

### Why does the CNN outperform the Dense model?

| Concept | Dense NN | CNN |
|:--|:--|:--|
| Input representation | Flat 784-vector — spatial structure lost | 28×28×1 tensor — full 2-D structure preserved |
| Feature extraction | Manual; all pixels connected to all neurons | Automatic via learnable convolutional filters |
| Translation invariance | ❌ None — shifting a digit changes every input | ✅ Partial — filters detect patterns regardless of position |
| Parameter efficiency | High (512×784 = 401K in first layer alone) | Lower (3×3×32 = 288 per filter, weight-sharing) |
| Hierarchical features | ❌ — single level of abstraction | ✅ — edges → curves → digit parts → digit identity |

### 🔑 Key Takeaways

1. **Spatial structure matters.** Flattening a 28×28 image to a 784-vector destroys the neighbourhood information between pixels. CNNs preserve and exploit that structure.

2. **Weight sharing makes CNNs efficient.** A 3×3 filter is applied across the entire image, so the same edge-detector is reused everywhere — giving far fewer parameters than a fully connected layer of equivalent reach.

3. **Hierarchical feature learning.** The first conv layer learns low-level edges, the second learns strokes and curves, and the dense head combines those into digit identities — matching the natural hierarchy of visual patterns.

4. **Overfitting risk.** With 10 epochs both models generalise well on MNIST. On harder datasets (CIFAR-10, ImageNet) the gap between Dense and CNN would be enormous, and regularisation (Dropout, Batch Normalisation) would become essential.

5. **Always check test accuracy, not just training accuracy.** Both models converge quickly on MNIST because it is a relatively simple dataset. Real-world image datasets require deeper architectures, data augmentation, and careful hyperparameter tuning.